# Estrategia de Adquisición de Datos — Análisis de Proveedores

Este notebook documenta y valida la arquitectura **dual-strategy** del `CompositeProvider`:

| Objetivo | Prioridad | Cadena |
|---|---|---|
| **NAV / Histórico** | ⚡ Velocidad | FMP → YFinance → MorningStar(light) |
| **Info / Sectores / Holdings** | 📊 Completitud | MorningStar(detailed) → FMP → Finect |

## Proveedores disponibles

| Proveedor | NAV | Info | Sectores | Países | Holdings | Comisiones | Velocidad NAV |
|-----------|-----|------|----------|--------|----------|------------|---------------|
| **FMP** | ✅ | ETFs | ✅ | ✅ | ✅ | ❌ | ~300ms |
| **Morningstar** | ✅ | Muy rica | ✅ | ✅ | ✅ | Parcial | ~2-5s |
| **Yahoo Finance** | ✅ | Mínima | ❌ | ❌ | ❌ | ❌ | ~500ms |
| **Finect** | ❌ | Rica | ❌* | ❌ | ❌* | ✅ Detallado | N/A |

*Holdings y asset allocation de Finect se renderizan vía JavaScript (no disponibles en scraping estático).

## Lógica de fusión
- **NAV**: primer proveedor que devuelve precio > 0 gana (failover)
- **Info**: se fusionan TODOS los proveedores (primer valor no-nulo de cada campo gana)
- **Sectores/Países/Holdings**: se usa la fuente con más datos (mayor granularidad)

## 0. Setup

In [5]:
import sys
import os
import importlib
import logging
import time
import pandas as pd
from IPython.display import display, Markdown

# Resolver rutas
if os.path.exists("../app"):
    sys.path.insert(0, os.path.abspath(".."))
    cache_path = "../data/cache"
else:
    sys.path.insert(0, os.path.abspath("./backend"))
    cache_path = "./backend/data/cache"

# Recargar módulos
import app.services.functions_fund
import app.services.data_providers
import app.services.finect_provider
import app.client
importlib.reload(app.services.functions_fund)
importlib.reload(app.services.data_providers)
importlib.reload(app.services.finect_provider)
importlib.reload(app.client)

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s", force=True)

# ISINs de prueba — portfolio real
TEST_ISINS = {
    "ES0141116030": "Hamco Global Value (RV Global Value)",
    "IE00BYX5NX33": "Fidelity MSCI World (RV Global Blend)",
    "ES0146309002": "Horos Value Internacional (RV Global Small Cap)",
    "LU1598719752": "Cobas Internacional (RV Global Small Cap)",
    "LU0840158819": "Storm Bond (Renta Fija)",
    "LU0302296495": "DNB Technology (RV Sector Tecnología)",
    "LU3038481936": "Hamco Global Value (RV Global Value)",
}

print(f"🔍 ISINs de prueba: {len(TEST_ISINS)}")
for isin, name in TEST_ISINS.items():
    print(f"   {isin} — {name}")

🔍 ISINs de prueba: 7
   ES0141116030 — Hamco Global Value (RV Global Value)
   IE00BYX5NX33 — Fidelity MSCI World (RV Global Blend)
   ES0146309002 — Horos Value Internacional (RV Global Small Cap)
   LU1598719752 — Cobas Internacional (RV Global Small Cap)
   LU0840158819 — Storm Bond (Renta Fija)
   LU0302296495 — DNB Technology (RV Sector Tecnología)
   LU3038481936 — Hamco Global Value (RV Global Value)


## 1. Verificación de cadenas del CompositeProvider

Comprobamos que las cadenas se construyen correctamente y que cada proveedor está disponible.

In [6]:
from app.services.data_providers import (
    CompositeProvider, FMPProvider, MStarProvider, YFinanceProvider
)
from app.services.finect_provider import FinectProvider, _get_finect_url

composite = CompositeProvider(cache_path=cache_path)

print("=" * 60)
print("ARQUITECTURA DEL CompositeProvider")
print("=" * 60)

print(f"\n⚡ Cadena NAV (velocidad):")
for i, p in enumerate(composite._nav_chain, 1):
    extra = ""
    if isinstance(p, FMPProvider):
        extra = f" [API key: {'✅' if p.available else '❌'}]"
    print(f"   {i}. {type(p).__name__}{extra}")

print(f"\n📊 Cadena Data (completitud):")
for i, p in enumerate(composite._data_chain, 1):
    extra = ""
    if isinstance(p, FMPProvider):
        extra = f" [API key: {'✅' if p.available else '❌'}]"
    print(f"   {i}. {type(p).__name__}{extra}")

print(f"\n📋 Todos los proveedores (sin duplicados): {[type(p).__name__ for p in composite.providers]}")

ARQUITECTURA DEL CompositeProvider

⚡ Cadena NAV (velocidad):
   1. FMPProvider [API key: ✅]
   2. YFinanceProvider
   3. MStarProvider

📊 Cadena Data (completitud):
   1. MStarProvider
   2. FMPProvider [API key: ✅]
   3. FinectProvider

📋 Todos los proveedores (sin duplicados): ['FMPProvider', 'YFinanceProvider', 'MStarProvider', 'FinectProvider']


## 2. Benchmark de velocidad — NAV

Medimos el tiempo de respuesta de cada proveedor para obtener el NAV de varios ISINs.
Esto valida que la cadena NAV está ordenada correctamente por velocidad.

In [7]:
fmp = FMPProvider()
yf_prov = YFinanceProvider()
mstar = MStarProvider(cache_path=cache_path)

results_nav = []

for isin, fund_name in TEST_ISINS.items():
    row = {"ISIN": isin, "Fondo": fund_name}
    
    # FMP
    t0 = time.time()
    try:
        nav = fmp.get_nav(isin) if fmp.available else None
    except Exception:
        nav = None
    row["FMP_NAV"] = nav
    row["FMP_ms"] = round((time.time() - t0) * 1000)
    
    # YFinance
    t0 = time.time()
    try:
        nav = yf_prov.get_nav(isin)
    except Exception:
        nav = None
    row["YF_NAV"] = nav
    row["YF_ms"] = round((time.time() - t0) * 1000)
    
    # MorningStar (solo caché para no bloquear)
    t0 = time.time()
    try:
        nav = mstar.get_nav(isin)
    except Exception:
        nav = None
    row["MStar_NAV"] = nav
    row["MStar_ms"] = round((time.time() - t0) * 1000)
    
    # Composite (cadena completa)
    t0 = time.time()
    try:
        nav = composite.get_nav(isin)
    except Exception:
        nav = None
    row["Composite_NAV"] = nav
    row["Composite_ms"] = round((time.time() - t0) * 1000)
    
    results_nav.append(row)
    print(f"  ✓ {isin} — FMP:{row['FMP_ms']}ms YF:{row['YF_ms']}ms MStar:{row['MStar_ms']}ms → Composite:{row['Composite_ms']}ms")

df_nav_bench = pd.DataFrame(results_nav)
print(f"\n{'='*60}")
print("RESULTADO: Benchmark NAV")
print(f"{'='*60}\n")
display(df_nav_bench[["Fondo", "FMP_NAV", "FMP_ms", "YF_NAV", "YF_ms", "MStar_NAV", "MStar_ms", "Composite_NAV", "Composite_ms"]])

Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0141116030 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0141116030 (light)
  ✓ ES0141116030 — FMP:0ms YF:1318ms MStar:44ms → Composite:6ms
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00BYX5NX33 (light)
  ✓ IE00BYX5NX33 — FMP:0ms YF:91ms MStar:27ms → Composite:35ms
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0146309002 (light)
  ✓ ES0146309002 — FMP:0ms YF:157ms MStar:29ms → Composite:39ms
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para LU1598719752 (light)
  ✓ LU1598719752 — FMP:0ms YF:82ms MStar:51ms → Composite:35ms
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para LU0840158819 (light)
  ✓ LU0840158819 — FMP:0ms YF:83ms MStar:50ms → Composite:41ms
Cache manager initialized with base

,Fondo,FMP_NAV,FMP_ms,YF_NAV,YF_ms,MStar_NAV,MStar_ms,Composite_NAV,Composite_ms
0,Hamco Global Value (RV Global Value),None,0,NaN,1318,NaN,44,NaN,6
1,Fidelity MSCI World (RV Global Blend),None,0,13.116800,91,13.116800,27,13.116800,35
2,Horos Value Internacional (RV Global Small Cap),None,0,214.738617,157,214.738617,29,214.738617,39
3,Cobas Internacional (RV Global Small Cap),None,0,175.389999,82,175.389999,51,175.389999,35
4,Storm Bond (Renta Fija),None,0,156.960007,83,156.960007,50,156.960007,41
5,DNB Technology (RV Sector Tecnología),None,0,1669.399048,75,1669.399048,416,1669.399048,29
6,Hamco Global Value (RV Global Value),None,0,282.556000,81,282.556000,40,282.556000,37


## 3. Benchmark de completitud — Fund Info

Medimos cuántos campos de información devuelve cada proveedor para validar la estrategia de fusión.

In [4]:
finect = FinectProvider()

results_info = []

for isin, fund_name in TEST_ISINS.items():
    row = {"ISIN": isin, "Fondo": fund_name}
    
    # FMP info
    t0 = time.time()
    try:
        info = fmp.get_fund_info(isin) if fmp.available else {}
    except Exception:
        info = {}
    row["FMP_campos"] = len(info)
    row["FMP_ms"] = round((time.time() - t0) * 1000)
    
    # MorningStar info
    t0 = time.time()
    try:
        info = mstar.get_fund_info(isin)
    except Exception:
        info = {}
    row["MStar_campos"] = len(info)
    row["MStar_ms"] = round((time.time() - t0) * 1000)
    
    # Finect info
    t0 = time.time()
    try:
        info = finect.get_fund_info(isin)
    except Exception:
        info = {}
    row["Finect_campos"] = len(info)
    row["Finect_ms"] = round((time.time() - t0) * 1000)
    
    # Composite info (fusión)
    t0 = time.time()
    try:
        info = composite.get_fund_info(isin)
    except Exception:
        info = {}
    row["Composite_campos"] = len(info)
    row["Composite_ms"] = round((time.time() - t0) * 1000)
    
    results_info.append(row)
    print(f"  ✓ {isin} — MStar:{row['MStar_campos']} FMP:{row['FMP_campos']} Finect:{row['Finect_campos']} → Composite:{row['Composite_campos']} campos")

df_info_bench = pd.DataFrame(results_info)
print(f"\n{'='*60}")
print("RESULTADO: Benchmark Info (campos por proveedor)")
print(f"{'='*60}\n")
display(df_info_bench[["Fondo", "MStar_campos", "MStar_ms", "FMP_campos", "FMP_ms", "Finect_campos", "Finect_ms", "Composite_campos", "Composite_ms"]])

Cache manager initialized with base path: ../data/cache

--- Procesando: IE00BYX5NX33 (detailed)---
✓ Datos obtenidos desde caché para IE00BYX5NX33
Obteniendo datos completos de Morningstar para IE00BYX5NX33...
Recopilando datos completos para IE00BYX5NX33...


selenium.webdriver.common.selenium_manager — Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


KeyboardInterrupt: 

## 4. Cobertura Finect — Resolución de URLs vía Sitemap

Finect requiere un *slug* en la URL (`/{ISIN}-{Nombre_slug}`). El módulo `finect_provider` descarga los sitemaps públicos (`funds.xml` + `etfs.xml`) y construye un índice ISIN→URL de ~52.000 fondos, cacheado en disco por 7 días.

In [ ]:
import json
from pathlib import Path

# Verificar el índice sitemap cacheado
cache_file = Path(cache_path).resolve().parent / "app" / "data" / "cache" / "finect_sitemap_index.json"
if not cache_file.exists():
    # Buscar en la ubicación relativa al notebook
    cache_file = Path("../app/data/cache/finect_sitemap_index.json").resolve()

if cache_file.exists():
    data = json.loads(cache_file.read_text(encoding="utf-8"))
    print(f"✅ Índice Finect cargado: {len(data):,} ISINs indexados")
    print(f"   Ubicación: {cache_file}")
    print(f"   Tamaño: {cache_file.stat().st_size / 1024:.0f} KB")
else:
    print("⚠️ Índice sitemap no encontrado en disco (se generará al primer uso)")
    data = {}

# Verificar cobertura de nuestro portfolio
print(f"\n{'='*60}")
print("COBERTURA del portfolio en Finect")
print(f"{'='*60}\n")

for isin, fund_name in TEST_ISINS.items():
    url = _get_finect_url(isin)
    if url:
        # Extraer solo el slug para display
        slug = url.split("/")[-1]
        print(f"  ✅ {isin} → {slug}")
    else:
        print(f"  ❌ {isin} — NO encontrado en Finect")

## 5. Detalle Finect — Campos extraídos de un fondo

Mostramos todos los campos que Finect aporta para un fondo de ejemplo.

In [ ]:
# Ejemplo con Cobas Internacional — fondo español con datos ricos en Finect
EXAMPLE_ISIN = "LU1598719752"

url = _get_finect_url(EXAMPLE_ISIN)
print(f"URL: {url}\n")

info = finect.get_fund_info(EXAMPLE_ISIN)
print(f"--- Finect devuelve {len(info)} campos para {EXAMPLE_ISIN} ---\n")
display(pd.DataFrame(list(info.items()), columns=["Campo", "Valor"]))

# Comparar con otro fondo
EXAMPLE_ISIN2 = "IE00BYX5NX33"
info2 = finect.get_fund_info(EXAMPLE_ISIN2)
print(f"\n--- Finect devuelve {len(info2)} campos para {EXAMPLE_ISIN2} ---\n")
display(pd.DataFrame(list(info2.items()), columns=["Campo", "Valor"]))

## 6. Sectores, Países y Holdings — Comparativa de completitud

Verificamos qué proveedor aporta más datos de cada tipo para cada fondo.

In [ ]:
results_detail = []

for isin, fund_name in TEST_ISINS.items():
    row = {"ISIN": isin, "Fondo": fund_name}
    
    # Sectores por proveedor
    try:
        s_mstar = mstar.get_sector_weights(isin)
    except Exception:
        s_mstar = {}
    try:
        s_fmp = fmp.get_sector_weights(isin) if fmp.available else {}
    except Exception:
        s_fmp = {}
    try:
        s_finect = finect.get_sector_weights(isin)
    except Exception:
        s_finect = {}
    
    row["Sect_MStar"] = len(s_mstar)
    row["Sect_FMP"] = len(s_fmp)
    row["Sect_Finect"] = len(s_finect)
    
    # Países por proveedor
    try:
        c_mstar = mstar.get_country_weights(isin)
    except Exception:
        c_mstar = {}
    try:
        c_fmp = fmp.get_country_weights(isin) if fmp.available else {}
    except Exception:
        c_fmp = {}
    
    row["País_MStar"] = len(c_mstar)
    row["País_FMP"] = len(c_fmp)
    
    # Holdings por proveedor
    try:
        h_mstar = mstar.get_holdings(isin)
    except Exception:
        h_mstar = pd.DataFrame()
    try:
        h_fmp = fmp.get_holdings(isin) if fmp.available else pd.DataFrame()
    except Exception:
        h_fmp = pd.DataFrame()
    try:
        h_finect = finect.get_holdings(isin)
    except Exception:
        h_finect = pd.DataFrame()
    
    row["Hold_MStar"] = len(h_mstar)
    row["Hold_FMP"] = len(h_fmp)
    row["Hold_Finect"] = len(h_finect)
    
    # Composite (lo que realmente obtiene el usuario)
    try:
        row["Sect_Composite"] = len(composite.get_sector_weights(isin))
    except Exception:
        row["Sect_Composite"] = 0
    try:
        row["País_Composite"] = len(composite.get_country_weights(isin))
    except Exception:
        row["País_Composite"] = 0
    try:
        row["Hold_Composite"] = len(composite.get_holdings(isin))
    except Exception:
        row["Hold_Composite"] = 0
    
    results_detail.append(row)

df_detail = pd.DataFrame(results_detail)

print(f"{'='*60}")
print("RESULTADO: Sectores por proveedor")
print(f"{'='*60}\n")
display(df_detail[["Fondo", "Sect_MStar", "Sect_FMP", "Sect_Finect", "Sect_Composite"]])

print(f"\n{'='*60}")
print("RESULTADO: Países/Regiones por proveedor")
print(f"{'='*60}\n")
display(df_detail[["Fondo", "País_MStar", "País_FMP", "País_Composite"]])

print(f"\n{'='*60}")
print("RESULTADO: Holdings por proveedor")
print(f"{'='*60}\n")
display(df_detail[["Fondo", "Hold_MStar", "Hold_FMP", "Hold_Finect", "Hold_Composite"]])

## 7. Fusión de Info — Campo por campo

Para un fondo de ejemplo, mostramos qué proveedor aporta cada campo en la fusión final.

In [ ]:
# Análisis campo por campo para un fondo de ejemplo
EXAMPLE = "IE00BYX5NX33"

info_sources = {}
try:
    info_sources["MorningStar"] = mstar.get_fund_info(EXAMPLE)
except Exception:
    info_sources["MorningStar"] = {}

try:
    info_sources["FMP"] = fmp.get_fund_info(EXAMPLE) if fmp.available else {}
except Exception:
    info_sources["FMP"] = {}

try:
    info_sources["Finect"] = finect.get_fund_info(EXAMPLE)
except Exception:
    info_sources["Finect"] = {}

info_sources["Composite"] = composite.get_fund_info(EXAMPLE)

# Construir tabla comparativa pivotada
all_fields = sorted(set().union(*[set(d.keys()) for d in info_sources.values()]))

rows = []
for field in all_fields:
    row = {"Campo": field}
    for provider_name, info_dict in info_sources.items():
        val = info_dict.get(field)
        if val is not None and str(val) != "" and str(val) != "nan":
            row[provider_name] = str(val)[:50]
        else:
            row[provider_name] = "—"
    rows.append(row)

df_fields = pd.DataFrame(rows).set_index("Campo")

print(f"{'='*60}")
print(f"FUSIÓN DE CAMPOS para {EXAMPLE}")
print(f"{'='*60}")
print(f"Total campos únicos entre todos los proveedores: {len(df_fields)}\n")
display(df_fields)

## 8. Resumen ejecutivo — Recomendaciones

Basado en los benchmarks anteriores, validamos las decisiones de diseño.

In [ ]:
# Resumen automático basado en los resultados
print("""
╔══════════════════════════════════════════════════════════════╗
║         RESUMEN EJECUTIVO — ESTRATEGIA DE PROVEEDORES       ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  CADENA NAV (⚡ velocidad):                                  ║
║  ┌─────────────┐    ┌───────────┐    ┌────────────────┐    ║
║  │ 1. FMP      │ →  │ 2.YFinance│ →  │ 3. MorningStar │    ║
║  │ ~300ms      │    │ ~500ms    │    │ ~2-5s (caché)  │    ║
║  │ 1 HTTP call │    │ No API key│    │ Backup final   │    ║
║  └─────────────┘    └───────────┘    └────────────────┘    ║
║                                                              ║
║  CADENA DATA (📊 completitud):                               ║
║  ┌────────────────┐  ┌──────────┐  ┌──────────────────┐   ║
║  │ 1. MorningStar │→ │ 2. FMP   │→ │ 3. Finect        │   ║
║  │ Ratings, risk  │  │ ETF data │  │ Comisiones, AUM  │   ║
║  │ Sectores, hold │  │ Sectors  │  │ Gestora, categ.  │   ║
║  └────────────────┘  └──────────┘  └──────────────────┘   ║
║                                                              ║
║  LÓGICA DE FUSIÓN:                                          ║
║  • get_fund_info(): merge ALL → primer valor no-nulo gana   ║
║  • get_sector/country/holdings(): fuente más completa gana  ║
║  • get_nav/history(): primer éxito = resultado              ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  FINECT — Datos exclusivos vía sitemap:                     ║
║  • 52.000 ISINs indexados (funds.xml + etfs.xml)            ║
║  • Caché JSON en disco (TTL: 7 días)                        ║
║  • Campos: gestora, categoría, benchmark, patrimonio,       ║
║    fecha inicio, comisiones detalladas (gestión, éxito,     ║
║    custodia, gastos corrientes, reembolso, suscripción)     ║
║  • Limitación: holdings y asset allocation son client-side  ║
╚══════════════════════════════════════════════════════════════╝
""")

# Estadísticas finales
if df_nav_bench is not None:
    nav_coverage = df_nav_bench["Composite_NAV"].notna().sum()
    print(f"📊 Cobertura NAV portfolio: {nav_coverage}/{len(TEST_ISINS)} ({nav_coverage/len(TEST_ISINS)*100:.0f}%)")
    avg_nav_ms = df_nav_bench["Composite_ms"].mean()
    print(f"⚡ Tiempo medio NAV (Composite): {avg_nav_ms:.0f}ms")

if df_info_bench is not None:
    avg_info = df_info_bench["Composite_campos"].mean()
    print(f"📊 Campos info medios (Composite): {avg_info:.0f}")
    finect_coverage = (df_info_bench["Finect_campos"] > 0).sum()
    print(f"🌐 Cobertura Finect: {finect_coverage}/{len(TEST_ISINS)} fondos")